In [13]:
import pandas as pd
import numpy as np
import altair as alt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

alt.data_transformers.disable_max_rows()

halibut = pd.read_csv(
    "../data/processed/halibut.csv"
)

roni_annual = pd.read_csv(
    "../data/processed/roni_annual.csv"
)

halibut_climate = pd.read_csv("../data/processed/halibut_climate.csv")

In [14]:
halibut_climate.head()

,Year,Pounds,Value,Annual_RONI
0,1980,726550.0,1000982.0,0.241667
1,1981,1262018.0,1874168.0,-0.208333
2,1982,1212312.0,1745829.0,1.175000
3,1983,1130363.0,1611535.0,0.475000
4,1984,1107019.0,1855602.0,-0.441667


In [15]:
alt.Chart(halibut_climate).mark_circle(size=80).encode(
    x=alt.X("Annual_RONI:Q", title="Annual Mean RONI"),
    y=alt.Y("Pounds:Q", title="California Halibut Landings (lbs)"),
    tooltip=["Year", "Annual_RONI", "Pounds"]
).properties(
    title="No-Lag Comparison: Annual RONI vs. California Halibut Landings",
    width=600,
    height=400
)

alt.Chart(...)

Is there an observable relationship between annual climate conditions (RONI) and annual California halibut landings when no lag is applied?

The scatterplot shows no obvious linear relationship between annual mean RONI and California halibut landings. Years with both high and low landings occur across nearly the full range of RONI values, suggesting that climate conditions alone do not explain annual fishery production.

California halibut require several years to grow before recruiting to the commercial fishery. Therefore, climate conditions during the landing year may not reflect the environmental conditions experienced by the fish entering the fishery. This observation motivates exploration of lagged climate relationships.

In [16]:
points = alt.Chart(halibut_climate).mark_circle(size=80).encode(
    x=alt.X("Annual_RONI:Q", title="Annual Mean RONI"),
    y=alt.Y("Pounds:Q", title="California Halibut Landings (lbs)"),
    tooltip=["Year", "Annual_RONI", "Pounds"]
)

trend = (
    alt.Chart(halibut_climate)
    .transform_regression("Annual_RONI", "Pounds")
    .mark_line(color="red")
    .encode(
        x="Annual_RONI:Q",
        y="Pounds:Q"
    )
)

points + trend

alt.LayerChart(...)

### Observation

Adding a regression line to the no-lag scatterplot suggests only a weak positive relationship between annual mean RONI and California halibut landings. The points remain widely scattered, indicating that same-year climate conditions are unlikely to fully explain variation in landings. This motivated the next step: testing whether climate conditions from prior years show a stronger relationship with landings.

In [21]:
correlations = []

for lag in range(0, 11):

    temp = halibut_climate.copy()

    temp["Lagged_RONI"] = temp["Annual_RONI"].shift(lag)

    corr = temp["Pounds"].corr(temp["Lagged_RONI"])

    correlations.append({
        "Lag": lag,
        "Correlation": corr
    })

corr_df = pd.DataFrame(correlations)

corr_df

,Lag,Correlation
0,0,0.107383
1,1,0.073498
2,2,0.140330
3,3,0.036910
4,4,0.166526
5,5,0.265917
6,6,0.306526
7,7,0.343820
8,8,0.233143
9,9,0.139797


In [22]:
alt.Chart(corr_df).mark_line(point=True).encode(
    x=alt.X("Lag:Q", title="Lag (Years)"),
    y=alt.Y("Correlation:Q", title="Pearson Correlation")
).properties(
    title="Correlation Between Annual RONI and California Halibut Landings by Lag",
    width=600,
    height=350
)

alt.Chart(...)

Exploratory Observation

The initial lag analysis suggested that the correlation between annual mean RONI and California halibut landings strengthened as longer lags were introduced, reaching its highest observed value at approximately 7 years. Rather than exhibiting a single isolated peak, the correlation increased gradually from approximately 4–7 years before declining, suggesting a biologically plausible delayed response between ocean climate conditions and commercial fishery landings.

This exploratory result prompted a review of the data preparation workflow. The original analysis limited the RONI dataset to the years 1980–2025, matching the temporal extent of the fisheries dataset. However, because lagged analyses associate earlier climate conditions with later fishery outcomes, climate observations from years preceding 1980 are also required. To preserve these relationships, the RONI dataset was expanded to include the complete available record (1950–2025) before applying temporal lags.

This refinement ensures that early years of the commercial fishery can be associated with the appropriate historical climate conditions and provides a more complete basis for evaluating delayed climate effects.

In [23]:
best_lag = 7

halibut_lag7 = halibut_climate.copy()

halibut_lag7["RONI_lag7"] = (
    halibut_lag7["Annual_RONI"]
    .shift(best_lag)
)

halibut_lag7 = halibut_lag7.dropna()

In [24]:
points = alt.Chart(halibut_lag7).mark_circle(size=80).encode(
    x=alt.X(
        "RONI_lag7:Q",
        title="Annual Mean RONI (7-Year Lag)"
    ),
    y=alt.Y(
        "Pounds:Q",
        title="California Halibut Landings (lbs)"
    ),
    tooltip=[
        "Year",
        "RONI_lag7",
        "Pounds"
    ]
)

trend = (
    alt.Chart(halibut_lag7)
    .transform_regression(
        "RONI_lag7",
        "Pounds"
    )
    .mark_line(color="red")
    .encode(
        x="RONI_lag7:Q",
        y="Pounds:Q"
    )
)

(points + trend).properties(
    title="Seven-Year Lag Relationship Between RONI and California Halibut Landings",
    width=650,
    height=400
)

alt.LayerChart(...)

To further investigate the lag analysis, annual California halibut landings were compared with annual mean RONI values shifted forward by seven years, corresponding to the strongest correlation identified in the previous analysis. Relative to the no-lag comparison, the regression line exhibited a steeper positive slope, indicating a stronger positive association between historical ocean climate conditions and subsequent commercial landings.

Although considerable variability remains, the seven-year lag relationship is noticeably stronger than the contemporaneous relationship, supporting the hypothesis that California halibut landings respond to ocean climate conditions after a biologically meaningful delay rather than within the same year.

An additional observation emerged from the lag correlation analysis. Rather than exhibiting a single sharp peak, the correlation increased gradually beginning at approximately four years, remained elevated through approximately seven years, and then declined. This broad response window is consistent with the biology of California halibut, where favorable environmental conditions may influence recruitment, growth, and eventual contribution to the commercial fishery over multiple years rather than at a single age or time point.

These exploratory findings suggest that incorporating lagged climate variables may improve the investigation of climate-fishery relationships and motivate the development of an interactive visualization that allows users to explore different lag periods across species.

In [25]:
time_series = halibut_climate.copy()

time_series["Landings_z"] = (
    time_series["Pounds"] -
    time_series["Pounds"].mean()
) / time_series["Pounds"].std()

time_series["RONI_z"] = (
    time_series["Annual_RONI"] -
    time_series["Annual_RONI"].mean()
) / time_series["Annual_RONI"].std()

time_series.head()

,Year,Pounds,Value,Annual_RONI,Landings_z,RONI_z
0,1980,726550.0,1000982.0,0.241667,-0.422031,0.402524
1,1981,1262018.0,1874168.0,-0.208333,1.394873,-0.264946
2,1982,1212312.0,1745829.0,1.175000,1.226215,1.786905
3,1983,1130363.0,1611535.0,0.475000,0.948152,0.748619
4,1984,1107019.0,1855602.0,-0.441667,0.868943,-0.611041


Time Series Exploration

While scatterplots and correlation coefficients quantify the strength of the relationship between climate conditions and commercial fishery landings, they do not illustrate how these variables change through time. To better understand the temporal dynamics of the relationship, annual California halibut landings and annual mean RONI were standardized using z-scores and plotted as time series. Standardization places both variables on a common scale, allowing their relative peaks and troughs to be compared despite their different units of measurement.

The first time series compares annual landings with contemporaneous (same-year) RONI values. The second repeats the comparison after applying the seven-year lag identified during the correlation analysis. Together, these visualizations provide a qualitative assessment of whether lagging the climate signal improves alignment between periods of favorable ocean conditions and subsequent fishery performance.

In [26]:
plot_df = time_series.melt(
    id_vars="Year",
    value_vars=["Landings_z", "RONI_z"],
    var_name="Variable",
    value_name="Z_score"
)

In [39]:
alt.Chart(plot_df).mark_line(point=True).encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y(
        "Z_score:Q",
        title="Standardized Value (z-score)"
    ),
    color=alt.Color(
        "Variable:N",
        title=""
    ),
    tooltip=["Year", "Variable", "Z_score"]
).properties(
    title="Annual California Halibut Landings and Annual Mean RONI",
    width=700,
    height=400
)

alt.Chart(...)

In [33]:
halibut_lag7.columns

Index(['Year', 'Pounds', 'Value', 'RONI', 'Landings_z', 'RONI_z'], dtype='str')

In [35]:
halibut_lag7["Landings_z"] = (
    halibut_lag7["Pounds"] - halibut_lag7["Pounds"].mean()
) / halibut_lag7["Pounds"].std()

halibut_lag7["RONI_z"] = (
    halibut_lag7["RONI"] - halibut_lag7["RONI"].mean()
) / halibut_lag7["RONI"].std()

In [36]:
time_series = halibut_lag7.melt(
    id_vars="Year",
    value_vars=["Landings_z", "RONI_z"],
    var_name="Variable",
    value_name="Z_score"
)

In [37]:
alt.Chart(time_series).mark_line(point=True).encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y("Z_score:Q", title="Standardized Value (z-score)"),
    color=alt.Color("Variable:N", title=""),
    tooltip=[
        "Year",
        "Variable",
        alt.Tooltip("Z_score:Q", format=".2f")
    ]
).properties(
    title="California Halibut Landings and Seven-Year Lagged RONI",
    width=750,
    height=400
)

alt.Chart(...)

Observations

The no-lag time series showed little visual correspondence between annual mean RONI and California halibut landings, consistent with the weak correlation observed in the initial scatterplot and regression analysis. After applying a seven-year lag to the climate index, periods of elevated and depressed RONI appeared to align more closely with subsequent increases and decreases in commercial landings. Although the correspondence is not exact, the lagged comparison provides additional visual evidence that California halibut landings are more closely associated with historical climate conditions than with climate conditions occurring in the same year.

The time series complements the previous statistical analyses by illustrating the temporal sequence underlying the lag hypothesis. Rather than suggesting an immediate response to changing ocean conditions, the visualization supports the interpretation that favorable climate conditions may influence recruitment, growth, and eventual availability to the commercial fishery over several years. This reinforces the biological plausibility of incorporating lagged climate variables when investigating climate–fishery relationships and provides a foundation for an interactive visualization in which users can explore species-specific lag effects.